# Fontes de Dados (cont'd)🚰 🎲

## ⚡️ Desafio: Raspagem Avançada de Dados ✂️

<pre>Utilizando os módulos <b>requests</b> e <b>bs4</b> (<b>Beautifulsoup</b>), crie uma função de nome <b>texto_wiki</b>, cujo único parâmetro é a URL de um verbete da Wikipedia (<i>e.g.</i> <a href="https://pt.wikipedia.org/wiki/Inteligência_artificial">https://pt.wikipedia.org/wiki/Inteligência_artificial</a>), que retornará o conteúdo do corpo principal da página do verbete.</pre>

👉 dica: analise os elementos da página HTML para identificar qual elemento deve ser selecionado para obter somente o texto principal do artigo.

👉 mais dica: use a documentação das bibliotecas <a href="https://requests.readthedocs.io/en/latest/">requests</b></a> e <a href="https://pypi.org/project/beautifulsoup4/">BeautifulSoup</a>.

<details>
  <summary>Resposta</summary>

<br/>

```python
import bs4
import requests

def texto_wiki(url=None):
    # acessar a página da Wikipedia
    pagina = requests.get(url)
    pagina.encoding = 'utf-8'

    # faz o parsing com o BeautifulSoup
    soup_page = bs4.BeautifulSoup(pagina.text, 'html.parser')

    # remove todas as tags que são úteis para navegação
    classes_a_ignorar = ['hatnote','navbox','vertical-navbox','reflist','mbox-text']
    ignorar = [item for item in soup_page.find_all(class_=classes_a_ignorar)]
    ignorar += soup_page.find_all(id='mw-panel-toc')
    [ref.extract() for ref in ignorar]
    
    #extrai o texto principal do verbete
    principal = soup_page.find('div', id='mw-content-text').get_text()
    texto = principal.replace('[editar | editar código-fonte]', '')
    return texto
```

<br/>

</details>

In [1]:
def texto_wiki(url_wiki):
    # escreva seu código aqui
    pass

<pre>Agora, teste sua função, executando o código abaixo.</pre>

In [2]:
url_wiki = 'https://pt.wikipedia.org/wiki/Inteligência_artificial'

ia_wiki = texto_wiki(url_wiki)
ia_wiki

<pre>Por fim, execute as duas células abaixo. É a combinação da nova função <b>texto_wiki</b> com os códigos do último exercício de Nível 3 para exibir as 20 palavras mais frequentes do texto do artigo, ignorando as palavras irrelevantes.</pre>

In [3]:
!pip install --upgrade nltk --quiet

In [4]:
import nltk
import re

def limpa_texto(linha):
    return ' '.join(re.sub(r'[^\w\s]', '', linha.lower() if type(linha) == str else '').split())

def calcula_frequencia(lista_itens=None):
    frequencia = dict()
    for item in lista_itens:
        frequencia[item] = frequencia.get(item, 0) + 1
    return frequencia

stopwords = nltk.corpus.stopwords.words('portuguese')
texto = texto_wiki(url_wiki)
palavras = limpa_texto(texto).split()
palavras_sem_stopwords = [palavra for palavra in palavras if palavra not in stopwords]
frequencia_de_palavras_sem_stopwords = calcula_frequencia(palavras_sem_stopwords)
palavras_sem_stopwords_mais_frequentes = sorted(frequencia_de_palavras_sem_stopwords.items(), key=lambda x: x[1], reverse=True)
for palavra_frequente_sem_ser_stopword in palavras_sem_stopwords_mais_frequentes[:20]:
    print(palavra_frequente_sem_ser_stopword[0], end='. ')

## Aspectos de API 📝

<pre>Algumas bases de dados do Governo Federal são disponibilizadas por meio de APIs que retornam dados em formato <b>JSON</b>.</pre>

<pre>O padrão <a href='https://swagger.io/specification/'>OpenAPI</a> tem sido utilizado por diversas organizações.</pre>

<pre>Um exemplo é a Secretaria do Tesouro Nacional, que disponibiliza dados do Portal de Custos do Governo Federal por meio da <a href="https://apidatalake.tesouro.gov.br/docs/custos/">API Custos</a>.</pre>

<pre>A API apresenta as seguintes restrições:
    - Retorno de até 250 linhas por página de resultado de pesquisa;
    - Limitação da taxa consumo dos dados: 1 consulta/segundo.</pre>

<pre>Acesse a página e teste a API pensionistas, verificando os parâmetros disponíveis para pesquisa:</pre>

👉 dica: informe, como parâmetros, o <b>mes</b> `1`, <b>ano</b> `2023` e <b>organizacao_n3</b> `047384`.

<img src="figs/pensionistas.png" width="600px" style="margin-left: auto; margin-right: auto;">

<pre>A saída da pesquisa acima em seu <a href="https://apidatalake.tesouro.gov.br/ords/custos/tt/pessoal_inativo?ano=2023&mes=1&organizacao_n3=047384">navegador</a> deverá ser próximo do <b>JSON</b> abaixo.</pre>

```json
{
	"items": [{
		"co_natureza_juridica": 3,
		"ds_natureza_juridica": "ADMINISTRACAO DIRETA",
		"co_organizacao_n0": "000026",
		"ds_organizacao_n0": null,
		"co_organizacao_n1": "267175",
		"ds_organizacao_n1": "MINISTERIO DA CIENCIA, TECNOLOGIA E INOVACAO",
		"co_organizacao_n2": "024753",
		"ds_organizacao_n2": "LABORATORIO NACIONAL DE ASTROFISICA",
		"co_organizacao_n3": "047384",
		"ds_organizacao_n3": "COORDENACAO DE ASTROFISICA",
		"an_lanc": 2023,
		"me_lanc": 1,
		"va_custo_pessoal_inativo": 23432.61
	}],
	"hasMore": false,
	"limit": 250,
	"offset": 0,
	"count": 1,
	"links": [{
		"rel": "self",
		"href": "https://apidatalake.tesouro.gov.br/ords/custos/tt/pessoal_inativo?ano=2023&mes=1&organizacao_n3=047384"
	}, {
		"rel": "describedby",
		"href": "https://apidatalake.tesouro.gov.br/ords/custos/metadata-catalog/tt/item"
	}, {
		"rel": "first",
		"href": "https://apidatalake.tesouro.gov.br/ords/custos/tt/pessoal_inativo?ano=2023&mes=1&organizacao_n3=047384"
	}]
}
```

<pre>Utilizando o método <b>request</b>, faça uma requisição à <b>API Pensionistas</b>, utilizando o método <b>get</b>, com a URL abaixo.</pre>

In [5]:
url = "https://apidatalake.tesouro.gov.br/ords/custos/tt/pensionistas"

<details>
  <summary>Resposta</summary>

<br/>

```python
import requests

resultado = requests.get(url)
```

<br/>

</details>

<pre>Agora, converta a resposta para um dicionário e liste as chaves do dicionário resultante.</pre>

👉 dica: use o método `json` na variável que armazena o valor de retorno do método `get` do `requests`.

<details>
  <summary>Resposta</summary>

<br/>

```python
dicionario = resultado.json()
dicionario.keys()
```

<br/>

</details>

<pre>Algumas observações:
    - Note que há uma chave denominada <b>hasMore</b> que indica se a pesquisa possui mais resultados além dos retornados incialmente;
    - Além disso, o parâmetro <b>offset</b> da URL deve ser incrementado em 250 para obtenção de novos resultados; e
    - Por fim, lembre-se da limitação de uma consulta por segundo.</pre>

<pre>Considerando as observações acima, crie uma função de nome <b>consomeAPI</b>, cujo único parâmetro é a URL da <b>API de Custos de Pessoal Inativo do Tesouro</b> (<i>e.g.</i> 'https://apidatalake.tesouro.gov.br/ords/custos/tt/pessoal_inativo?ano=2023&mes=1'), que retornará uma lista contendo todos os itens de resultado da pesquisa.</pre>

👉 dica: para atender ao requisito da limitação de tempo a cada consulta, pesquise sobre o método `sleep` da biblioteca `time` do Python.

<details>
  <summary>Resposta</summary>

<br/>

```python
import requests
import time

def consome_API(url):
    dados = []
    offset = 0
    hasMore = True
    
    while hasMore:
        resultado = requests.get(url, params={'offset': offset})
        dicionario = resultado.json()
        dados += dicionario['items']
        hasMore = dicionario['hasMore']
        offset += 250
        time.sleep(1)
        
    return dados
```

<br/>

</details>

In [6]:
def consome_API(url):
    # escreva seu código aqui
    pass

<pre>Agora, teste sua função, executando o código abaixo.</pre>

In [7]:
url_pesquisa = 'https://apidatalake.tesouro.gov.br/ords/custos/tt/pessoal_inativo?ano=2023&mes=1'

inativos_202301 = consome_API(url_pesquisa)
inativos_202301

<pre>Sabendo que o custo é armazenado no campo <b>va_custo_pessoal_inativo</b> do dicionário <b>inativos_202301</b>, calcule o valor total dessa despesa para o mês de janeiro de 2023, segundo os dados retornados pela API.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
despesa = sum(item['va_custo_pessoal_inativo'] for item in inativos_202301)

f'R$ {despesa:,.2f}'
```

<br/>

</details>

<pre>Agora, utilize a função <b>consome_API</b> para baixar os dados de todos os meses do período entre 2020 a 2023.</pre>

<pre>Armazene o resultado na variável <b>despesas_inativos</b>.</pre>

<details>
  <summary>Resposta</summary>

<br/>

```python
despesas_inativos = []

for ano in range (2020, 2024):
    for mes in range(1, 13):
        despesas_inativos += consome_API(f'https://apidatalake.tesouro.gov.br/ords/custos/tt/pessoal_inativo?ano={ano}&mes={mes}')
```

<br/>

</details>

<pre>Por fim, armazene a base de dados no arquivo <b>despesas_inativos_2015-22.json</b>, no formato de <b>json</b>.</pre>

👉 dica: importe a biblioteca <a href="https://docs.python.org/pt-br/3/library/json.html">json</a> e utilize o método `dump` da biblioteca para gravar os dados.

<details>
  <summary>Resposta</summary>

<br/>

```python
import json

with open("despesas_inativos_2015-22.json", "w") as json_file:
    json.dump(despesas_inativos, json_file)
```

<br/>

</details>